# Tiga cara menjawab, satu dataset — flat vs 12 sigmoid vs 5 kepala Category

Notebook ini menjalankan `ml_workspace/yolo/train_variants.py`: tiga model dengan backbone, data,
split, dan augmentasi yang sama, hanya lapisan terakhirnya berbeda.

| varian | cara menjawab |
| --- | --- |
| `flat` | satu softmax 12 kelas — desain yang terpasang sekarang, jadi pembanding |
| `sigmoid` | 12 pengecek ya/tidak yang berbagi satu backbone ("belasan classifier" versi praktis) |
| `heads` | satu softmax per Category ditambah gerbang pemilih Category |

Yang dicari bukan cuma top-1: laporan juga mencatat cakupan (berapa persen crop dijawab), akurasi
saat menjawab, macro-F1, dan recall per kelas, karena Rompi hanya punya 76 crop uji.

**Sebelum Run All:** panel kanan → Accelerator **GPU T4**, Internet **On**. Perkiraan 25–40 menit
per varian; total sekitar 1,5 jam termasuk pemotongan crop.


In [ ]:
# Langkah 1/7 — cek runtime.
!pip install -q ultralytics "datasets>=3" "huggingface_hub>=0.23"

import sys
import torch
import ultralytics

print(f"python        {sys.version.split()[0]}")
print(f"torch         {torch.__version__}")
print(f"ultralytics   {ultralytics.__version__}")
print(f"cuda          {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu           {torch.cuda.get_device_name(0)}")
else:
    print("              panel kanan -> Accelerator -> GPU T4, lalu jalankan ulang dari sel ini")


## 2. Ambil repo

Skripnya ada di repo, bukan di notebook: `fashionpedia_crops.py` memotong crop, `train_variants.py`
melatih ketiga varian dan menulis laporan. Ganti `REPO_REF` hanya kalau skripnya ada di branch lain.


In [ ]:
# Langkah 2/7 — clone repo pada ref yang memuat train_variants.py.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kmilaz/gayaku.git"
REPO_REF = "feat/train-variants"
REPO_DIR = Path("/kaggle/temp/gayaku")


def run(command, **kwargs):
    print("+", " ".join(str(part) for part in command))
    completed = subprocess.run(command, text=True, **kwargs)
    if completed.returncode != 0:
        raise SystemExit(f"perintah gagal ({completed.returncode}): {' '.join(map(str, command))}")
    return completed


if not (REPO_DIR / ".git").is_dir():
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)])
CROPS = REPO_DIR / "ml_workspace/scripts/fashionpedia_crops.py"
VARIANTS = REPO_DIR / "ml_workspace/yolo/train_variants.py"
for path in (CROPS, VARIANTS):
    if not path.is_file():
        raise SystemExit(f"{path} tidak ada di {REPO_REF}; ganti REPO_REF ke branch yang memuatnya")
print("repo siap", REPO_DIR)


## 3. Unduh Fashionpedia dan potong crop

Tujuh berkas parquet anotasi. `fashionpedia_crops.py` memotong setiap kotak item dengan margin 5 %
dan membatasi **2.000 crop per kelas** dibagi 80/10/10, dipisah **per foto** agar satu orang tidak
masuk dua split. Pemotongan ini sama dengan run yang menghasilkan model terpasang, jadi angkanya
sebanding.


In [ ]:
# Langkah 3/7 — unduh dan potong crop.
from huggingface_hub import snapshot_download

HF_DIR = Path("/kaggle/temp/fashionpedia_hf")
CROP_DIR = Path("/kaggle/temp/crops")
PER_CLASS = 2000

if not list((HF_DIR / "data").glob("*.parquet")) or not (HF_DIR / "dataset_infos.json").is_file():
    snapshot_download(
        repo_id="detection-datasets/fashionpedia",
        repo_type="dataset",
        local_dir=str(HF_DIR),
        allow_patterns=["data/*.parquet", "dataset_infos.json"],
    )
print(f"parquet   {len(list((HF_DIR / 'data').glob('*.parquet')))} berkas di {HF_DIR}")

if not (CROP_DIR / "manifest.json").is_file():
    run(
        [sys.executable, str(CROPS), "--hf-dir", str(HF_DIR), "--out", str(CROP_DIR),
         "--per-class", str(PER_CLASS)],
        cwd=str(REPO_DIR),
    )
print("crop siap", CROP_DIR)


## 4. Periksa pohon sebelum menghabiskan kuota GPU

Empat pemeriksaan, semuanya berhenti kalau gagal: 12 kelas persis, tidak ada kelas kosong di split
mana pun, dan tidak ada satu foto pun yang muncul di dua split.


In [ ]:
# Langkah 4/7 — periksa manifest.
MANIFEST = json.loads((CROP_DIR / "manifest.json").read_text(encoding="utf-8"))
CLASSES = MANIFEST["classes"]
if len(CLASSES) != 12:
    raise SystemExit(f"kelasnya {len(CLASSES)}, bukan 12: {CLASSES}")
for split in ("train", "val", "test"):
    missing = [label for label in CLASSES if not MANIFEST["splits"][split].get(label)]
    if missing:
        raise SystemExit(f"split {split} tidak punya contoh untuk: {missing}")

print(f"{'kelas':22s}{'latih':>8}{'val':>8}{'uji':>8}")
for label in CLASSES:
    print(f"{label:22s}{MANIFEST['splits']['train'][label]:>8}"
          f"{MANIFEST['splits']['val'][label]:>8}{MANIFEST['splits']['test'][label]:>8}")
total = {split: sum(MANIFEST["splits"][split].values()) for split in ("train", "val", "test")}
print("total", total)


## 5. Latih ketiga varian

`train_variants.py` memilih epoch terbaik di `val` (patience 8), lalu membuka `test` sekali. Semua
varian memakai seed, optimizer, dan augmentasi yang sama, jadi perbedaan angka berasal dari cara
menjawab. Varian `heads` juga mencetak `heads_oracle`: Category diambil dari label sebenarnya,
sebagai batas atas yang tidak bisa dipakai di aplikasi.


In [ ]:
# Langkah 5/7 — latih dan ukur.
EPOCHS = 30
BATCH = 64
PATIENCE = 8
OUT = REPO_DIR / "ml_workspace/yolo/artifacts/variants"

run(
    [sys.executable, str(VARIANTS),
     "--data", str(CROP_DIR),
     "--out", str(OUT),
     "--epochs", str(EPOCHS),
     "--batch", str(BATCH),
     "--patience", str(PATIENCE),
     "--workers", "2"],
    cwd=str(REPO_DIR),
)


## 6. Baca laporannya

Tabel ini yang dipakai memutuskan. Cara membacanya: `top-1 uji` adalah yang dibandingkan dengan
angka terpasang **0,6784**; `cakupan` dan `akurasi saat menjawab` menunjukkan apa yang didapat kalau
model boleh berkata "tidak dikenali"; kolom per kelas menunjukkan siapa yang menang dan kalah.


In [ ]:
# Langkah 6/7 — tampilkan laporan.
from IPython.display import Markdown, display

display(Markdown((OUT / "variants_report.md").read_text(encoding="utf-8")))


## 7. Bungkus hasilnya

Zip berisi laporan, `variants.json`, dan confusion matrix tiap varian. Unduh dari panel Output,
lalu lampirkan angkanya ke thread proyek sebelum ada keputusan mengganti model.


In [ ]:
# Langkah 7/7 — zip artefak.
import zipfile

ZIP = Path("/kaggle/working/gayaku_variants.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(OUT.parent))
print(ZIP, f"{ZIP.stat().st_size / 1024:.0f} KB")
